![Stencils](svg/swm_composite.svg)

In [ ]:
from gt4py import next as gtx
from gt4py.next.experimental import concat_where

I = gtx.Dimension("I")
J = gtx.Dimension("J")

dtype = gtx.float64
IJField = gtx.Field[gtx.Dims[I, J], dtype]


@gtx.field_operator
def avg_x(f: IJField):
    """Average field in the x direction."""
    return 0.5 * (f(I + 1) + f)


@gtx.field_operator
def avg_y(f: IJField):
    """Average field in the y direction."""
    return 0.5 * (f(J + 1) + f)


@gtx.field_operator
def avg_x_staggered(f: IJField):
    """Average field which is staggered in x in the x direction."""
    return 0.5 * (f(I - 1) + f)


@gtx.field_operator
def avg_y_staggered(f: IJField):
    """Average field which is staggered in y in the y direction."""
    return 0.5 * (f(J - 1) + f)


@gtx.field_operator
def delta_x(dx: dtype, f: IJField):
    """Calculate the difference in the x direction."""
    return (1.0 / dx) * (f(I + 1) - f)


@gtx.field_operator
def delta_y(dx: dtype, f: IJField):
    """Calculate the difference in the y direction."""
    return (1.0 / dx) * (f(J + 1) - f)


@gtx.field_operator
def delta_x_staggered(dx: dtype, f: IJField):
    """Calculate the difference in the x direction for field staggered in x."""
    return (1.0 / dx) * (f - f(I - 1))


@gtx.field_operator
def delta_y_staggered(dx: dtype, f: IJField):
    """Calculate the difference in the y direction for field staggered in y."""
    return (1.0 / dx) * (f - f(J - 1))


@gtx.field_operator
def make_periodic(f: IJField, M: gtx.int32, N: gtx.int32):
    """Make the field f periodic by copying values from the opposite sides."""
    f = concat_where(I == -1, f(I + M), f)
    f = concat_where(I == M, f(I - M), f)
    f = concat_where(J == -1, f(J + N), f)
    f = concat_where(J == N, f(J - N), f)
    return f


@gtx.field_operator
def timestep(
    u: IJField,
    v: IJField,
    p: IJField,
    dx: dtype,
    dy: dtype,
    dt: dtype,
    uold: IJField,
    vold: IJField,
    pold: IJField,
    alpha: dtype,
    M: gtx.int32,
    N: gtx.int32,
) -> tuple[IJField, IJField, IJField, IJField, IJField, IJField]:
    cu = avg_x(p) * u
    cv = avg_y(p) * v
    z = (delta_x(dx, v) - delta_y(dy, u)) / avg_x(avg_y(p))
    h = p + 0.5 * (avg_x_staggered(u * u) + avg_y_staggered(v * v))

    unew = uold + avg_y_staggered(z) * avg_y_staggered(avg_x(cv)) * dt - delta_x(dx, h) * dt
    vnew = vold - avg_x_staggered(z) * avg_x_staggered(avg_y(cu)) * dt - delta_y(dy, h) * dt
    pnew = pold - delta_x_staggered(dx, cu) * dt - delta_y_staggered(dy, cv) * dt

    uold_new = u + alpha * (unew - 2.0 * u + uold)
    vold_new = v + alpha * (vnew - 2.0 * v + vold)
    pold_new = p + alpha * (pnew - 2.0 * p + pold)

    unew = make_periodic(unew, M, N)
    vnew = make_periodic(vnew, M, N)
    pnew = make_periodic(pnew, M, N)

    uold_new = make_periodic(uold_new, M, N)
    vold_new = make_periodic(vold_new, M, N)
    pold_new = make_periodic(pold_new, M, N)

    return (
        unew,
        vnew,
        pnew,
        uold_new,
        vold_new,
        pold_new,
    )

In [ ]:
import initial_conditions
from jax import numpy as jnp
import numpy as np


allocator = jnp

dt0 = 0.0
dt = 90.0
dt = dt
dx = 100000.0
dy = 100000.0
fsdx = 4.0 / (dx)
fsdy = 4.0 / (dy)
a = 1000000.0
alpha = 0.001

M = 16
N = 16

domain = gtx.domain({I: (-1, M + 1), J: (-1, N + 1)})

# Initialize fields
_u, _v, _p = initial_conditions.initialize_2halo(np, M, N, dx, dy, a)
u = gtx.as_field(domain, _u, dtype=dtype, allocator=allocator)
v = gtx.as_field(domain, _v, dtype=dtype, allocator=allocator)
p = gtx.as_field(domain, _p, dtype=dtype, allocator=allocator)

# Initial old fields
uold = gtx.as_field(domain, _u, dtype=dtype, allocator=allocator)
vold = gtx.as_field(domain, _v, dtype=dtype, allocator=allocator)
pold = gtx.as_field(domain, _p, dtype=dtype, allocator=allocator)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output

vlims = {
    "u": (_u.min(), _u.max()),
    "v": (_v.min(), _v.max()),
    "p": (_p.min(), _p.max()),
}


def live_plot(u, v, p):
    clear_output(wait=True)
    fields = [u.asnumpy(), v.asnumpy(), p.asnumpy()]
    titles = ["u", "v", "p"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, data, title in zip(axes, fields, titles):
        vmin, vmax = vlims[title]
        im = ax.imshow(data, origin="lower", vmin=vmin, vmax=vmax)
        ax.set_title(title)
        fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


live_plot(u, v, p)

In [ ]:
import jax
import functools

prog = jax.jit(functools.partial(timestep.definition, M=M, N=N))

for ncycle in range(100):
    unew, vnew, pnew, uold, vold, pold = prog(
        u=u,
        v=v,
        p=p,
        dx=dx,
        dy=dy,
        dt=dt if ncycle == 0 else dt * 2.0,
        uold=uold,
        vold=vold,
        pold=pold,
        alpha=alpha if ncycle > 0 else 0.0,
    )

    # swap x with xnew fields
    u, unew = unew, u
    v, vnew = vnew, v
    p, pnew = pnew, p

    if ncycle % 1 == 0:
        live_plot(
            u,
            v,
            p,
        )